# 🤖 Analyse de Sentiment — From Scratch vs Transfer Learning
## RNCP37827 — Développeur en Intelligence Artificielle

Ce notebook compare deux approches d'analyse de sentiment :
1. **Approche classique (from scratch)** : TF-IDF + algorithmes de Machine Learning (Naive Bayes, Logistic Regression, SVM)
2. **Transfer Learning** : DistilBERT pré-entraîné (HuggingFace)

**Dataset** : SST-2 (Stanford Sentiment Treebank) — 67 000 phrases en anglais


## 1. Installation des dépendances

In [ ]:
# Installation des bibliothèques nécessaires
# (décommenter si pas encore installées)
# !pip install datasets transformers torch scikit-learn pandas matplotlib seaborn wordcloud tqdm


In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ── Data & ML classique ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
import re
import time

# ── Sklearn ──
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
)

# ── HuggingFace ──
from datasets import load_dataset
from transformers import pipeline as hf_pipeline

print("✅ Imports réussis")
print(f"   pandas {pd.__version__} | numpy {np.__version__}")
import sklearn; print(f"   scikit-learn {sklearn.__version__}")
import transformers; print(f"   transformers {transformers.__version__}")


## 2. Collecte des données

### 2.1 Chargement du dataset SST-2 (HuggingFace Datasets)

SST-2 (Stanford Sentiment Treebank v2) est le dataset standard pour l'analyse de sentiment binaire.
- **Taille** : ~67 000 phrases d'entraînement, ~872 de validation
- **Labels** : 0 = NEGATIVE, 1 = POSITIVE
- **Source** : critiques de films (Rotten Tomatoes)


In [ ]:
# Chargement depuis HuggingFace Hub
print("Chargement du dataset SST-2...")
dataset = load_dataset("glue", "sst2")

print(f"\nStructure du dataset :")
print(dataset)
print(f"\nExemples d'entraînement : {len(dataset['train'])}")
print(f"Exemples de validation   : {len(dataset['validation'])}")


In [ ]:
# Conversion en DataFrames pandas
df_train = pd.DataFrame(dataset['train'])
df_val   = pd.DataFrame(dataset['validation'])

# Renommer les colonnes pour plus de clarté
df_train = df_train.rename(columns={'sentence': 'texte', 'label': 'sentiment'})
df_val   = df_val.rename(columns={'sentence': 'texte', 'label': 'sentiment'})

# Convertir les labels numériques en texte
df_train['label_texte'] = df_train['sentiment'].map({0: 'NEGATIVE', 1: 'POSITIVE'})
df_val['label_texte']   = df_val['sentiment'].map({0: 'NEGATIVE', 1: 'POSITIVE'})

print("=== Aperçu des données d'entraînement ===")
print(df_train.head(8).to_string(index=False))
print(f"\nShape train : {df_train.shape}")
print(f"Shape val   : {df_val.shape}")


### 2.2 Chargement des données du projet (CSV local)

In [ ]:
# Charger les données locales du projet (si disponibles)
import os

chemin_csv = 'data/sample_reviews.csv'

if os.path.exists(chemin_csv):
    df_projet = pd.read_csv(chemin_csv)
    print(f"✅ CSV projet chargé : {len(df_projet)} avis")
    print(df_projet.head())
else:
    # Créer un mini-dataset de démonstration
    print("⚠️  CSV projet non trouvé — création d'un dataset de démonstration")
    df_projet = pd.DataFrame({
        'texte': [
            "Ce produit est absolument fantastique, je le recommande vivement !",
            "Très déçu, qualité médiocre, ne correspond pas à la description.",
            "Livraison rapide, produit conforme, parfait rapport qualité/prix.",
            "Complètement cassé à la réception, service client inexistant.",
            "Excellent achat, je suis très satisfait de cette commande.",
        ],
        'note': [5, 1, 4, 1, 5],
        'sentiment_manuel': ['POSITIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE']
    })
    print(df_projet)


## 3. Exploration et nettoyage des données (EDA)

### 3.1 Analyse exploratoire

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── Distribution des labels ──
label_counts = df_train['label_texte'].value_counts()
colors = ['#EF4444', '#22C55E']
axes[0].bar(label_counts.index, label_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribution des labels\n(Train set)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Nombre d\'exemples')
for i, (label, count) in enumerate(label_counts.items()):
    axes[0].text(i, count + 200, f'{count:,}\n({count/len(df_train)*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')

# ── Distribution de la longueur des textes ──
df_train['longueur'] = df_train['texte'].str.len()
df_train['nb_mots'] = df_train['texte'].str.split().str.len()

for label, color in zip(['NEGATIVE', 'POSITIVE'], ['#EF4444', '#22C55E']):
    subset = df_train[df_train['label_texte'] == label]['nb_mots']
    axes[1].hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='white')
axes[1].set_title('Distribution du nombre de mots\npar sentiment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Nombre de mots')
axes[1].set_ylabel('Fréquence')
axes[1].legend()

# ── Mots les plus fréquents ──
all_words = ' '.join(df_train['texte'].tolist()).lower().split()
stop_words = {'the', 'a', 'an', 'is', 'it', 'this', 'and', 'of', 'to', 'in', 'that', 'for', 'as', 's', 'with'}
words_filtered = [w for w in all_words if w not in stop_words and len(w) > 2]
word_freq = Counter(words_filtered).most_common(15)
words, freqs = zip(*word_freq)
axes[2].barh(words[::-1], freqs[::-1], color='#6366F1', edgecolor='white')
axes[2].set_title('15 mots les plus fréquents\n(hors stop words)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Fréquence')

plt.suptitle('Analyse Exploratoire — Dataset SST-2', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n📊 Statistiques du dataset :")
print(f"  Longueur moyenne des textes  : {df_train['longueur'].mean():.0f} caractères")
print(f"  Nombre de mots moyen         : {df_train['nb_mots'].mean():.1f} mots")
print(f"  Texte le plus court          : {df_train['nb_mots'].min()} mots")
print(f"  Texte le plus long           : {df_train['nb_mots'].max()} mots")


### 3.2 Pipeline de nettoyage du texte

In [ ]:
def nettoyer_texte(texte: str) -> str:
    """
    Pipeline de nettoyage du texte :
    1. Mise en minuscules
    2. Suppression des caractères spéciaux (garder les lettres et espaces)
    3. Suppression des espaces multiples
    4. Strip (suppression des espaces en début/fin)
    """
    texte = texte.lower()
    texte = re.sub(r'[^a-zA-Z\s]', ' ', texte)       # garder seulement lettres + espaces
    texte = re.sub(r'\s+', ' ', texte)                 # espaces multiples → 1 espace
    texte = texte.strip()
    return texte

# Appliquer le nettoyage
df_train['texte_clean'] = df_train['texte'].apply(nettoyer_texte)
df_val['texte_clean']   = df_val['texte'].apply(nettoyer_texte)

# Vérification qualité
textes_vides = (df_train['texte_clean'].str.len() == 0).sum()
print(f"Textes vides après nettoyage : {textes_vides}")

# Supprimer les textes vides si nécessaire
df_train = df_train[df_train['texte_clean'].str.len() > 0].reset_index(drop=True)
df_val   = df_val[df_val['texte_clean'].str.len() > 0].reset_index(drop=True)

# Aperçu avant/après
print("\n=== Avant / Après nettoyage ===")
for i in range(3):
    print(f"Avant  : {df_train['texte'].iloc[i]}")
    print(f"Après  : {df_train['texte_clean'].iloc[i]}")
    print()


In [ ]:
# Sous-échantillonnage pour l'entraînement (pour que l'exécution soit rapide)
# On garde 10 000 exemples d'entraînement (5000 pos + 5000 neg)
TAILLE_TRAIN = 10_000
TAILLE_VAL   = 872  # toute la validation

df_train_sample = (
    df_train.groupby('sentiment', group_keys=False)
            .apply(lambda x: x.sample(n=TAILLE_TRAIN//2, random_state=42))
            .reset_index(drop=True)
)

print(f"✅ Données préparées :")
print(f"   Train : {len(df_train_sample)} exemples ({df_train_sample['sentiment'].value_counts().to_dict()})")
print(f"   Val   : {len(df_val)} exemples ({df_val['sentiment'].value_counts().to_dict()})")

X_train = df_train_sample['texte_clean'].tolist()
y_train = df_train_sample['sentiment'].tolist()
X_val   = df_val['texte_clean'].tolist()
y_val   = df_val['sentiment'].tolist()


## 4. Modèles From Scratch — TF-IDF + Machine Learning

### Principe
Le TF-IDF (Term Frequency – Inverse Document Frequency) transforme chaque texte en **vecteur numérique**.
Il mesure l'importance d'un mot dans un document par rapport à l'ensemble du corpus.

```
TF(t, d)  = nombre d'occurrences du terme t dans le document d
IDF(t)    = log(N / nombre de documents contenant t)
TF-IDF(t, d) = TF(t, d) × IDF(t)
```

Les vecteurs TF-IDF sont ensuite passés à des algorithmes de classification classiques.


### 4.1 Naive Bayes (Multinomial)

In [ ]:
# ── Modèle 1 : Naive Bayes ──
# Principe : calcule P(classe | mots) avec le théorème de Bayes
# Très rapide, bon sur du texte, fonctionne bien avec peu de données

pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),    # unigrammes ET bigrammes
        min_df=2,              # ignorer les mots très rares
        sublinear_tf=True,     # log(tf) pour lisser les fréquences
    )),
    ('clf', ComplementNB(alpha=0.1)),
])

print("Entraînement Naive Bayes...")
t0 = time.time()
pipeline_nb.fit(X_train, y_train)
temps_train_nb = time.time() - t0
print(f"✅ Entraîné en {temps_train_nb:.2f}s")

# Évaluation
t0 = time.time()
y_pred_nb = pipeline_nb.predict(X_val)
temps_inf_nb = (time.time() - t0) / len(X_val) * 1000  # ms par exemple

acc_nb = accuracy_score(y_val, y_pred_nb)
f1_nb  = f1_score(y_val, y_pred_nb)
print(f"\n📊 Naive Bayes :")
print(f"   Accuracy  : {acc_nb:.4f} ({acc_nb*100:.2f}%)")
print(f"   F1-Score  : {f1_nb:.4f}")
print(f"   Latence   : {temps_inf_nb:.3f} ms/exemple")


### 4.2 Régression Logistique

In [ ]:
# ── Modèle 2 : Régression Logistique ──
# Principe : apprend des poids pour chaque mot (feature) afin de maximiser P(y|x)
# Très interprétable : on peut voir quels mots pèsent le plus

pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(
        C=5.0,             # régularisation (plus C est grand, moins de régularisation)
        max_iter=1000,
        solver='lbfgs',
        n_jobs=-1,
    )),
])

print("Entraînement Logistic Regression...")
t0 = time.time()
pipeline_lr.fit(X_train, y_train)
temps_train_lr = time.time() - t0
print(f"✅ Entraîné en {temps_train_lr:.2f}s")

# Évaluation
t0 = time.time()
y_pred_lr = pipeline_lr.predict(X_val)
temps_inf_lr = (time.time() - t0) / len(X_val) * 1000

acc_lr = accuracy_score(y_val, y_pred_lr)
f1_lr  = f1_score(y_val, y_pred_lr)
print(f"\n📊 Logistic Regression :")
print(f"   Accuracy  : {acc_lr:.4f} ({acc_lr*100:.2f}%)")
print(f"   F1-Score  : {f1_lr:.4f}")
print(f"   Latence   : {temps_inf_lr:.3f} ms/exemple")

# Mots les plus importants
tfidf_vocab = pipeline_lr.named_steps['tfidf'].get_feature_names_out()
coefs = pipeline_lr.named_steps['clf'].coef_[0]
top_pos = [(tfidf_vocab[i], coefs[i]) for i in coefs.argsort()[-15:]][::-1]
top_neg = [(tfidf_vocab[i], coefs[i]) for i in coefs.argsort()[:15]]

print("\n🟢 Top mots POSITIFS :", [w for w, _ in top_pos[:8]])
print("🔴 Top mots NÉGATIFS :", [w for w, _ in top_neg[:8]])


### 4.3 SVM Linéaire

In [ ]:
# ── Modèle 3 : SVM Linéaire ──
# Principe : trouve l'hyperplan qui maximise la marge entre les deux classes
# Souvent le meilleur algorithme classique pour la classification de texte

pipeline_svm = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
    )),
    ('clf', LinearSVC(
        C=1.0,
        max_iter=2000,
        dual=False,
    )),
])

print("Entraînement SVM Linéaire...")
t0 = time.time()
pipeline_svm.fit(X_train, y_train)
temps_train_svm = time.time() - t0
print(f"✅ Entraîné en {temps_train_svm:.2f}s")

# Évaluation
t0 = time.time()
y_pred_svm = pipeline_svm.predict(X_val)
temps_inf_svm = (time.time() - t0) / len(X_val) * 1000

acc_svm = accuracy_score(y_val, y_pred_svm)
f1_svm  = f1_score(y_val, y_pred_svm)
print(f"\n📊 SVM Linéaire :")
print(f"   Accuracy  : {acc_svm:.4f} ({acc_svm*100:.2f}%)")
print(f"   F1-Score  : {f1_svm:.4f}")
print(f"   Latence   : {temps_inf_svm:.3f} ms/exemple")


### 4.4 Rapport détaillé (meilleur modèle from scratch)

In [ ]:
# Identifier le meilleur modèle from scratch
modeles_scratch = {
    'Naive Bayes': (y_pred_nb, acc_nb),
    'Logistic Regression': (y_pred_lr, acc_lr),
    'SVM Linéaire': (y_pred_svm, acc_svm),
}
meilleur_nom = max(modeles_scratch, key=lambda k: modeles_scratch[k][1])
y_pred_best_scratch, _ = modeles_scratch[meilleur_nom]

print(f"Meilleur modèle from scratch : {meilleur_nom}")
print("\n" + classification_report(y_val, y_pred_best_scratch,
                                    target_names=['NEGATIVE', 'POSITIVE']))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_val, y_pred_best_scratch)
disp = ConfusionMatrixDisplay(cm, display_labels=['NEGATIVE', 'POSITIVE'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matrice de confusion\n{meilleur_nom}', fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Transfer Learning — DistilBERT

### Principe du Transfer Learning

Le Transfer Learning consiste à réutiliser un modèle pré-entraîné sur une grande quantité de données,
puis à l'appliquer (ou l'affiner) sur notre tâche spécifique.

```
Pré-entraînement (coûteux, fait une fois) :
  BERT/DistilBERT entraîné sur 11 milliards de mots (Wikipedia + BooksCorpus)
  → Apprend la structure du langage, les relations entre mots
                          ↓
Fine-tuning (rapide, notre étape) :
  Ajout d'une couche de classification sur SST-2 (~67K exemples)
  → Le modèle apprend la tâche spécifique (sentiment)
                          ↓
Inférence (notre usage dans le projet) :
  Modèle distilbert-base-uncased-finetuned-sst-2-english
  → Prêt à l'emploi, 91% de précision
```

**DistilBERT** est une version allégée de BERT :
- 66 millions de paramètres (vs 110M pour BERT-base)
- 97% des performances de BERT, 60% de la taille
- 60% plus rapide à l'inférence


In [ ]:
# Chargement du modèle DistilBERT via HuggingFace pipeline
# Modèle : distilbert-base-uncased-finetuned-sst-2-english
# Ce modèle a déjà été fine-tuné sur SST-2 → pas besoin de réentraîner

print("Chargement de DistilBERT (téléchargement si première fois ~250Mo)...")
t0 = time.time()

modele_bert = hf_pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,          # -1 = CPU, 0 = GPU si disponible
    truncation=True,
    max_length=512,
)

print(f"✅ Modèle chargé en {time.time() - t0:.1f}s")

# Test rapide
test = modele_bert("This movie is absolutely fantastic!")
print(f"\nTest : 'This movie is absolutely fantastic!'")
print(f"  → {test[0]['label']} (score: {test[0]['score']:.4f})")


In [ ]:
# Évaluation sur le jeu de validation
# Note : on limite à 500 exemples pour la vitesse (l'inférence BERT est plus lente)
TAILLE_EVAL_BERT = 500

X_val_bert = X_val[:TAILLE_EVAL_BERT]
y_val_bert  = y_val[:TAILLE_EVAL_BERT]

print(f"Évaluation sur {TAILLE_EVAL_BERT} exemples...")
print("(Cela peut prendre 1-2 minutes sur CPU...)")

t0 = time.time()

# Prédictions en batch pour accélérer
resultats_bert = modele_bert(X_val_bert, batch_size=32)

temps_total_bert = time.time() - t0
temps_inf_bert = temps_total_bert / TAILLE_EVAL_BERT * 1000  # ms par exemple

# Convertir les prédictions au format numérique (0=NEG, 1=POS)
y_pred_bert_label = [r['label'] for r in resultats_bert]
y_pred_bert = [1 if label == 'POSITIVE' else 0 for label in y_pred_bert_label]
scores_bert = [r['score'] for r in resultats_bert]

acc_bert = accuracy_score(y_val_bert, y_pred_bert)
f1_bert  = f1_score(y_val_bert, y_pred_bert)

print(f"\n📊 DistilBERT (Transfer Learning) :")
print(f"   Accuracy  : {acc_bert:.4f} ({acc_bert*100:.2f}%)")
print(f"   F1-Score  : {f1_bert:.4f}")
print(f"   Latence   : {temps_inf_bert:.1f} ms/exemple")
print(f"   Temps total ({TAILLE_EVAL_BERT} exemples) : {temps_total_bert:.1f}s")


In [ ]:
# Rapport détaillé DistilBERT
print(classification_report(y_val_bert, y_pred_bert,
                             target_names=['NEGATIVE', 'POSITIVE']))

# Distribution des scores de confiance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Matrice de confusion
cm_bert = confusion_matrix(y_val_bert, y_pred_bert)
ConfusionMatrixDisplay(cm_bert, display_labels=['NEGATIVE', 'POSITIVE']).plot(
    ax=axes[0], colorbar=False, cmap='Purples')
axes[0].set_title('Matrice de confusion\nDistilBERT', fontweight='bold')

# Distribution des scores de confiance
pos_scores = [scores_bert[i] for i in range(len(y_pred_bert)) if y_pred_bert[i] == 1]
neg_scores = [scores_bert[i] for i in range(len(y_pred_bert)) if y_pred_bert[i] == 0]
axes[1].hist(pos_scores, bins=30, alpha=0.7, color='#22C55E', label='POSITIVE')
axes[1].hist(neg_scores, bins=30, alpha=0.7, color='#EF4444', label='NEGATIVE')
axes[1].set_title('Distribution des scores de confiance\nDistilBERT', fontweight='bold')
axes[1].set_xlabel('Score de confiance')
axes[1].set_ylabel('Fréquence')
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Comparaison des performances

### 6.1 Tableau récapitulatif


In [ ]:
# Évaluer les modèles from scratch sur le même sous-ensemble que BERT
y_pred_nb_sub  = pipeline_nb.predict(X_val_bert)
y_pred_lr_sub  = pipeline_lr.predict(X_val_bert)
y_pred_svm_sub = pipeline_svm.predict(X_val_bert)

resultats = {
    'Modèle': ['Naive Bayes', 'Logistic Regression', 'SVM Linéaire', 'DistilBERT (TL)'],
    'Approche': ['From Scratch', 'From Scratch', 'From Scratch', 'Transfer Learning'],
    'Accuracy': [
        accuracy_score(y_val_bert, y_pred_nb_sub),
        accuracy_score(y_val_bert, y_pred_lr_sub),
        accuracy_score(y_val_bert, y_pred_svm_sub),
        acc_bert,
    ],
    'F1-Score': [
        f1_score(y_val_bert, y_pred_nb_sub),
        f1_score(y_val_bert, y_pred_lr_sub),
        f1_score(y_val_bert, y_pred_svm_sub),
        f1_bert,
    ],
    'Latence (ms/ex)': [temps_inf_nb, temps_inf_lr, temps_inf_svm, temps_inf_bert],
    'Taille modèle': ['~1 MB', '~8 MB', '~8 MB', '~255 MB'],
    'Temps entraîn.': [
        f'{temps_train_nb:.1f}s',
        f'{temps_train_lr:.1f}s',
        f'{temps_train_svm:.1f}s',
        'Pré-entraîné (67h GPU)',
    ],
}

df_resultats = pd.DataFrame(resultats)
df_resultats['Accuracy (%)'] = (df_resultats['Accuracy'] * 100).round(2)
df_resultats['F1 (%)'] = (df_resultats['F1-Score'] * 100).round(2)

print("=" * 80)
print(df_resultats[['Modèle', 'Approche', 'Accuracy (%)', 'F1 (%)',
                     'Latence (ms/ex)', 'Taille modèle', 'Temps entraîn.']].to_string(index=False))
print("=" * 80)


### 6.2 Visualisation comparative

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Comparaison : From Scratch vs Transfer Learning', fontsize=15, fontweight='bold', y=1.02)

modeles = df_resultats['Modèle'].tolist()
couleurs = ['#6366F1', '#8B5CF6', '#A855F7', '#F97316']
approches = df_resultats['Approche'].tolist()

# ── Accuracy ──
bars = axes[0].bar(modeles, df_resultats['Accuracy (%)'], color=couleurs, edgecolor='white', linewidth=1.5)
axes[0].set_title('Accuracy (%)', fontsize=13, fontweight='bold')
axes[0].set_ylim(70, 100)
axes[0].set_ylabel('Accuracy (%)')
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, df_resultats['Accuracy (%)']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)
# Ligne de référence
axes[0].axhline(y=90, color='green', linestyle='--', alpha=0.4, label='90% (seuil production)')
axes[0].legend(fontsize=9)

# ── F1-Score ──
bars = axes[1].bar(modeles, df_resultats['F1 (%)'], color=couleurs, edgecolor='white', linewidth=1.5)
axes[1].set_title('F1-Score (%)', fontsize=13, fontweight='bold')
axes[1].set_ylim(70, 100)
axes[1].set_ylabel('F1-Score (%)')
axes[1].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, df_resultats['F1 (%)']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)

# ── Latence (log scale) ──
latences = df_resultats['Latence (ms/ex)'].tolist()
bars = axes[2].bar(modeles, latences, color=couleurs, edgecolor='white', linewidth=1.5)
axes[2].set_title('Latence par exemple (ms)\n(échelle log)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Latence (ms)')
axes[2].set_yscale('log')
axes[2].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, latences):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.5,
                 f'{val:.2f}ms', ha='center', fontsize=9)

# Légende approches
patch_scratch = mpatches.Patch(color='#8B5CF6', label='From Scratch (TF-IDF)')
patch_tl      = mpatches.Patch(color='#F97316', label='Transfer Learning (BERT)')
fig.legend(handles=[patch_scratch, patch_tl], loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, -0.05), fontsize=11, frameon=True)

plt.tight_layout()
plt.savefig('comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphique sauvegardé : comparaison_modeles.png")


### 6.3 Analyse des erreurs

In [ ]:
# Analyser les exemples mal classifiés par le meilleur modèle from scratch et par BERT
y_pred_lr_sub_arr = np.array(y_pred_lr_sub)
y_pred_bert_arr   = np.array(y_pred_bert)
y_val_bert_arr    = np.array(y_val_bert)

# Cas intéressants : LR se trompe mais BERT a raison
lr_faux_bert_vrai = np.where((y_pred_lr_sub_arr != y_val_bert_arr) &
                              (y_pred_bert_arr == y_val_bert_arr))[0]

# Cas où les deux se trompent
les_deux_faux = np.where((y_pred_lr_sub_arr != y_val_bert_arr) &
                          (y_pred_bert_arr != y_val_bert_arr))[0]

print("=== Exemples où LR échoue mais BERT réussit ===\n")
for idx in lr_faux_bert_vrai[:5]:
    vrai_label = 'POSITIVE' if y_val_bert_arr[idx] == 1 else 'NEGATIVE'
    lr_label   = 'POSITIVE' if y_pred_lr_sub_arr[idx] == 1 else 'NEGATIVE'
    bt_label   = y_pred_bert_label[idx]
    print(f"Texte   : {X_val_bert[idx][:80]}...")
    print(f"  Vrai  : {vrai_label} | LR : {lr_label} ❌ | BERT : {bt_label} ✅")
    print()

print(f"\n=== Exemples difficiles (les deux se trompent) ===\n")
for idx in les_deux_faux[:3]:
    vrai_label = 'POSITIVE' if y_val_bert_arr[idx] == 1 else 'NEGATIVE'
    print(f"Texte  : {X_val_bert[idx][:80]}...")
    print(f"  Vrai : {vrai_label} | LR : ❌ | BERT : ❌")
    print()


### 6.4 Test sur des exemples personnalisés

In [ ]:
# Tester les modèles sur des phrases personnalisées
phrases_test = [
    "This film is a masterpiece, absolutely brilliant!",
    "Terrible movie, complete waste of time and money.",
    "Not bad, but could have been much better.",      # cas ambigu
    "I was really disappointed by this product.",
    "Outstanding quality, exceeded all my expectations!",
    "The worst experience I've ever had.",
]

print(f"{'Texte':<55} {'Vrai ?':>5} {'Naive B.':>10} {'Log.Reg.':>10} {'SVM':>8} {'BERT':>12}")
print("─" * 110)

for phrase in phrases_test:
    phrase_clean = nettoyer_texte(phrase)

    pred_nb  = 'POS' if pipeline_nb.predict([phrase_clean])[0] == 1 else 'NEG'
    pred_lr  = 'POS' if pipeline_lr.predict([phrase_clean])[0] == 1 else 'NEG'
    pred_svm = 'POS' if pipeline_svm.predict([phrase_clean])[0] == 1 else 'NEG'

    res_bert    = modele_bert(phrase)[0]
    pred_bert   = 'POS' if res_bert['label'] == 'POSITIVE' else 'NEG'
    score_bert  = res_bert['score']

    # Accord entre les 4 modèles
    accord = '✅' if pred_nb == pred_lr == pred_svm == pred_bert else '⚠️'

    print(f"{phrase[:54]:<55} {accord:>5} {pred_nb:>10} {pred_lr:>10} {pred_svm:>8} {pred_bert}({score_bert:.2f})")


## 7. Conclusions

### 7.1 Résumé des résultats


In [ ]:
# Tableau récapitulatif final
print("=" * 90)
print(f"{'SYNTHÈSE COMPARATIVE — Analyse de Sentiment':^90}")
print("=" * 90)
print(f"{'Modèle':<25} {'Accuracy':>10} {'F1-Score':>10} {'Latence':>12} {'Cas d\'usage':>30}")
print("─" * 90)

cas_usage = {
    'Naive Bayes':         'Baseline rapide, peu de données',
    'Logistic Regression': 'Production légère, interprétable',
    'SVM Linéaire':        'Meilleur from scratch, robuste',
    'DistilBERT (TL)':     'Production haute précision',
}

for _, row in df_resultats.iterrows():
    nom = row['Modèle']
    print(f"{nom:<25} {row['Accuracy (%)']:>9.2f}% {row['F1 (%)']:>9.2f}% "
          f"{row['Latence (ms/ex)']:>10.2f}ms  {cas_usage.get(nom, ''):>30}")

print("=" * 90)


### 7.2 Quand utiliser chaque approche ?

| Approche | ✅ Avantages | ❌ Inconvénients | 📌 Quand l'utiliser |
|----------|------------|-----------------|---------------------|
| **Naive Bayes** | Ultra rapide, pas de GPU | Moins précis, sensible au vocabulaire | Prototype, très peu de données |
| **Logistic Regression** | Interprétable, rapide, bon équilibre | Dépend de la qualité du TF-IDF | Production légère, < 100ms requis |
| **SVM Linéaire** | Souvent le meilleur classique | Moins interprétable que LR | Classification text standard |
| **DistilBERT** | Meilleure précision, comprend le contexte, gère les nuances | Lent (CPU), lourd (255 Mo), "boîte noire" | Cas où la précision prime sur la vitesse |

### 7.3 Recommandation pour le projet

Pour l'**Analyseur de Sentiment d'Avis Clients** :
- **DistilBERT** est le choix retenu pour la précision (~91%)
- Si la latence devient un problème (volume > 1000 req/s), basculer vers **SVM Linéaire** (~85-87%)
- Pour une version française : remplacer par **CamemBERT** (même architecture, entraîné sur du français)

### 7.4 Notion de Transfer Learning — Résumé pédagogique

```
Sans Transfer Learning (from scratch) :
  TF-IDF → vecteur de fréquences (chaque mot = une dimension)
  Modèle → apprend des règles simples sur les fréquences de mots
  Limite → ne comprend pas "not bad" = positif

Avec Transfer Learning (DistilBERT) :
  Tokenizer → sous-mots (ex: "unforgettable" → ["un", "forget", "##table"])
  Embeddings → vecteurs 768 dimensions par token (contextuels !)
  Attention → "not bad" → le modèle sait que "not" modifie "bad"
  Résultat → ~91% de précision vs ~85-88% pour les approches classiques
```


## 8. XGBoost — Optimisation (GridSearch), métriques, learning curve et MLflow

Cette section ajoute un **quatrième modèle** : **XGBoost** (gradient boosting d'arbres),
appliqué sur les vecteurs TF-IDF. On y met en oeuvre les bonnes pratiques de ML attendues
pour la certification :

1. **GridSearchCV** : recherche systématique des meilleurs hyperparamètres
2. **Métriques de classification** : accuracy, precision, recall, F1, ROC-AUC, matrice de confusion
3. **Courbe d'apprentissage** (learning curve) : diagnostic sur/sous-apprentissage
4. **MLflow** : suivi des expériences (paramètres + métriques + modèle)
5. **Tests pytest** : `tests/test_xgboost_sentiment.py`

> La logique cœur est dans le module réutilisable `ml/xgboost_sentiment.py`, ce qui la rend
> testable automatiquement avec pytest. Le notebook ne fait que l'appeler et visualiser.

In [ ]:
# Import du module XGBoost du projet
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from ml import xgboost_sentiment as xs
print('MLflow disponible :', xs.MLFLOW_DISPONIBLE)

# On réutilise les données déjà nettoyées plus haut (X_train, y_train, X_val, y_val).
# XGBoost + GridSearch étant coûteux, on sous-échantillonne pour garder un temps raisonnable.
import numpy as np
rng = np.random.RandomState(42)

N_TRAIN_XGB = 4000   # augmentez si vous avez le temps
N_VAL_XGB   = 1000

idx_tr = rng.choice(len(X_train), size=min(N_TRAIN_XGB, len(X_train)), replace=False)
idx_va = rng.choice(len(X_val),   size=min(N_VAL_XGB,   len(X_val)),   replace=False)

X_tr_xgb = [X_train[i] for i in idx_tr]
y_tr_xgb = [int(y_train[i]) for i in idx_tr]
X_va_xgb = [X_val[i] for i in idx_va]
y_va_xgb = [int(y_val[i]) for i in idx_va]

print(f"Train XGBoost : {len(X_tr_xgb)} | Val XGBoost : {len(X_va_xgb)}")

### 8.1 GridSearchCV — recherche des meilleurs hyperparamètres

On explore une petite grille (profondeur des arbres, nombre d'arbres, taux d'apprentissage)
en validation croisée 3 plis, optimisée sur le **F1-score**.

In [ ]:
import time

param_grid = {
    'clf__n_estimators': [200, 400],
    'clf__max_depth':    [4, 6],
    'clf__learning_rate':[0.1, 0.3],
}

print("GridSearchCV en cours (cela peut prendre 1-3 min)...")
t0 = time.time()
grid = xs.run_gridsearch(X_tr_xgb, y_tr_xgb, param_grid=param_grid, cv=3, scoring='f1')
print(f"Terminé en {time.time()-t0:.1f}s")

print("\nMeilleurs hyperparamètres :", grid.best_params_)
print(f"Meilleur F1 (CV)          : {grid.best_score_:.4f}")

model_xgb = grid.best_estimator_

### 8.2 Métriques de classification (jeu de validation)

In [ ]:
metrics = xs.evaluate_classification(model_xgb, X_va_xgb, y_va_xgb)

print(f"Accuracy  : {metrics['accuracy']:.4f}")
print(f"Precision : {metrics['precision']:.4f}")
print(f"Recall    : {metrics['recall']:.4f}")
print(f"F1-score  : {metrics['f1']:.4f}")
print(f"ROC-AUC   : {metrics['roc_auc']:.4f}" if metrics['roc_auc'] is not None else "ROC-AUC : n/a")
print("\n--- Rapport de classification ---")
print(metrics['classification_report'])

# Matrice de confusion
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 4))
xs.plot_confusion(metrics['confusion_matrix'], target_names=['NEGATIVE', 'POSITIVE'], ax=ax)
plt.tight_layout(); plt.show()

### 8.3 Courbe d'apprentissage (learning curve)

La courbe d'apprentissage montre l'évolution du score (entraînement vs validation croisée)
quand on augmente la taille du jeu d'entraînement. Un écart important entre les deux courbes
indique du **sur-apprentissage** ; des scores bas et proches indiquent du **sous-apprentissage**.

In [ ]:
lc = xs.compute_learning_curve(
    model_xgb, X_tr_xgb, y_tr_xgb,
    cv=3, scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 5),
)

fig, ax = plt.subplots(figsize=(7, 5))
xs.plot_learning_curve(lc, ax=ax)
plt.tight_layout(); plt.show()

### 8.4 Suivi de l'expérience avec MLflow

`train_with_mlflow()` relance la recherche, évalue le modèle et **enregistre dans MLflow** :
hyperparamètres, métriques, rapport de classification et modèle sérialisé.

Backend de suivi : base **SQLite** (`mlflow.db`), recommandée par MLflow.
Pour visualiser les runs, lancez dans un terminal :

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
puis ouvrez http://localhost:5000

In [ ]:
if xs.MLFLOW_DISPONIBLE:
    resultat = xs.train_with_mlflow(
        X_tr_xgb, y_tr_xgb, X_va_xgb, y_va_xgb,
        param_grid={'clf__n_estimators': [300], 'clf__max_depth': [6], 'clf__learning_rate': [0.1]},
        cv=3,
        experiment_name='sentiment-xgboost',
        run_name='xgboost-notebook',
        tracking_uri='sqlite:///mlflow.db',
    )
    print('Run MLflow enregistré :', resultat['run_id'])
    print('Meilleurs params      :', resultat['best_params'])
    print('Métriques            :', {k: round(v, 4) for k, v in resultat['metrics'].items()
                                       if isinstance(v, (int, float))})
    print('\nVisualisez avec : mlflow ui --backend-store-uri sqlite:///mlflow.db')
else:
    print("MLflow n'est pas installé. Installez-le : pip install mlflow")

### 8.5 Tests automatisés (pytest)

Le module `ml/xgboost_sentiment.py` est couvert par `tests/test_xgboost_sentiment.py`
(pipeline, métriques, GridSearch, learning curve, entraînement + MLflow).

```bash
pytest tests/test_xgboost_sentiment.py -v
```

In [ ]:
print("🎉 Notebook terminé !")
print("\nFichiers générés :")
import os
if os.path.exists('comparaison_modeles.png'):
    print("  ✅ comparaison_modeles.png")
